# 21 · What we know about the scoring mistakes, and why a person has to read some of them

**Purpose.** Notebooks 18-20 and the plans since asked whether better weights, richer features, a
better concept source or a better match key would let a learned score beat the hand-set X2 on the A2
final set. None did. This notebook (1) collects what has been measured, (2) shows where the mistakes
sit, using the error-analysis-2 records, and (3) explains what those numbers **cannot** tell us, which
is why the owner's manual review (D57) is part of the plan.

**Status.** `exploratory`. Every figure is read from a committed record in `output/extraction/`; this
notebook defines no model. The A2 final set is used to diagnose only and nothing is fitted on it (D46).
D50 is the development set (A2 train region, LLM labels). No resume or JD text appears here.

A *mistake* is a (Fit, No Fit) pair inside one JD that the score orders the wrong way; the
*inversion rate* is the share of such pairs that are misordered (a tie counts half).

In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from candidate_screener import reporting
from candidate_screener.config import OUTPUT

reporting.style(plt)
rec = lambda name: json.load(open(OUTPUT / "extraction" / name))
ea = rec("error-analysis-2.json")
soft, exact_r, exact_rn = rec("concept-stage2-R_N-soft0.60.json"), rec("concept-stage2-R.json"), rec("concept-stage2-R_N.json")
overlap, variants, thr = rec("concept-overlap.json"), rec("coverage-variants.json"), rec("match-threshold.json")
A2, D50S, D50H = "A2 final (59 JDs)", "D50 selection (70 JDs)", "D50 held-out (30 JDs)"

## 1 · What has been tried, and the best A2 result of each

Best = the largest paired total ΔAUC against the hand-set X2 (0.850) among that record's final fits.
`bar met` is the registered bar: CI above 0 **and** `generic` and `targeted` both positive. A pass
counts only if the same learner meets it on the A1-LLM and the D50 fit.

In [2]:
def best(final):
    top = max(final, key=lambda r: r["delta_total"])
    return pd.Series({"best Δ total": top["delta_total"], "CI": str([round(c, 3) for c in top["ci_total"]]),
                      "fits meeting the bar": sum(r["bar_met"] for r in final), "fits": len(final)})
tried = pd.DataFrame({
    "seed-rule concepts (R)": best(exact_r["final"]),
    "+ MiniLM concepts (R+N), exact match": best(exact_rn["final"]),
    "R+N, soft match at 0.60": best(soft["soft"]["final"])}).T
tried.round(3)

,best Δ total,CI,fits meeting the bar,fits
seed-rule concepts (R),0.005823,"[-0.009, 0.021]",0,6
"+ MiniLM concepts (R+N), exact match",0.004736,"[-0.012, 0.022]",0,6
"R+N, soft match at 0.60",0.004668,"[-0.027, 0.035]",0,6


In [3]:
g = overlap["sets"]["A2 final"]
pd.DataFrame({"no-overlap pairs": {a: s["no_overlap_share"] for a, s in g["stats"].items()},
              "cut vs R (pp)": {a: round(v, 1) for a, v in g["gate"]["cut_pp"].items()}}).round(3)

,no-overlap pairs,cut vs R (pp)
G,0.777,-24.3
N,0.645,-11.2
R,0.533,NaN
R+G,0.530,0.3
R+N,0.475,5.8


Reading: the seed-rule vocabulary left half of the A2 pairs without any concept overlap, a tagger
raised overlap, and a soft match key raised it further and lifted the coverage feature on its own.
**None of it moved the full score.** Coverage-only AUC and the full score are different things: role
and TF-IDF already carry most of what the extra matches add. The weighting variants (W1-W3) and the
dilution fix (W2b) also did not help, inside or outside the score (`coverage-variants.json`).

## 2 · Where the mistakes sit

### 2.1 Relative to the JD's role family
*Role* is the role model's probability that the CV is in the JD's role family; >= 0.5 counts as in role.

In [4]:
KINDS = list(ea["sets"][A2]["hand-set X2"]["share_by_kind"])
rows = {}
for s in (A2, D50S):
    r = ea["sets"][s]["hand-set X2"]
    for k in KINDS:
        rows[(k, s)] = {"share of mistakes": r["share_by_kind"][k], "inversion rate": r["rate_by_kind"][k]}
pd.DataFrame(rows).T.unstack(1).round(2)

share of mistakes                         \
                           A2 final (59 JDs) D50 selection (70 JDs)   
Fit in role, No Fit out                 0.01                   0.01   
Fit out of role, No Fit in              0.09                   0.14   
both in role                            0.20                   0.42   
both out of role                        0.71                   0.43   

                              inversion rate                         
                           A2 final (59 JDs) D50 selection (70 JDs)  
Fit in role, No Fit out                 0.00                   0.00  
Fit out of role, No Fit in              0.94                   0.89  
both in role                            0.42                   0.37  
both out of role                        0.20                   0.22

In [5]:
r = ea["sets"][A2]["hand-set X2"]
display(Markdown(
    f"On the A2 final set the hand-set X2 misorders **{r['inversion_rate']:.1%}** of its {r['pairs']:,} (Fit, No Fit) pairs. "
    f"**{r['share_by_kind']['both out of role']:.0%}** of the mistakes are between two CVs that the role model puts out of the JD's role "
    f"(role gives no signal there), and when both CVs are in role the score gets **{r['rate_by_kind']['both in role']:.0%}** of pairs wrong, "
    f"the worst rate. Pairs where a Fit CV sits out of role and a No Fit CV in role are rare "
    f"({r['share_by_kind']['Fit out of role, No Fit in']:.0%} of mistakes) but almost always wrong "
    f"({r['rate_by_kind']['Fit out of role, No Fit in']:.0%}). D50 has more same-family pairs (80%), so its mix is different."))

On the A2 final set the hand-set X2 misorders **14.8%** of its 1,143 (Fit, No Fit) pairs. **71%** of the mistakes are between two CVs that the role model puts out of the JD's role (role gives no signal there), and when both CVs are in role the score gets **42%** of pairs wrong, the worst rate. Pairs where a Fit CV sits out of role and a No Fit CV in role are rare (9% of mistakes) but almost always wrong (94%). D50 has more same-family pairs (80%), so its mix is different.

### 2.2 Which score term pushed the wrong way hardest
For each mistake, the term with the largest `weight × (x_NoFit - x_Fit)`. This is an **arithmetic
attribution**, not a cause: a term that varies a lot will often be the largest push.

In [6]:
cul = pd.DataFrame({(s, w): ea["sets"][s][w]["share_by_culprit"] for s in (A2, D50S, D50H) for w in ("hand-set X2", "logistic base (4)")}).T
cul.round(2)

experience_compatible  role_match  \
A2 final (59 JDs)      hand-set X2                         0.12        0.33   
                       logistic base (4)                   0.05        0.35   
D50 selection (70 JDs) hand-set X2                         0.06        0.35   
                       logistic base (4)                   0.02        0.41   
D50 held-out (30 JDs)  hand-set X2                         0.06        0.46   
                       logistic base (4)                   0.03        0.55   

                                          tfidf_scaled  coverage  
A2 final (59 JDs)      hand-set X2                0.56       NaN  
                       logistic base (4)          0.48      0.12  
D50 selection (70 JDs) hand-set X2                0.59       NaN  
                       logistic base (4)          0.46      0.11  
D50 held-out (30 JDs)  hand-set X2                0.48       NaN  
                       logistic base (4)          0.36      0.06

### 2.3 What a perfect role signal or cleaner labels would be worth
*Role oracle*: `role_match` replaced by the same-Primary-Keyword flag (metadata, forbidden as a system
input by D53; used here as a ceiling only). *Label tiers*: per-JD AUC on the pairs whose final label came
from two agreeing humans, one human, or an LLM tie-break. The pair sets differ per tier, so this is a
comparison across subsets, not a paired delta.

In [7]:
ab = ea["ablations_a2"]
o = pd.DataFrame({w: {"role oracle Δ total": a["role_oracle_total"]["delta"],
                      "CI": str([round(c, 3) for c in a["role_oracle_total"]["ci"]]),
                      "Δ generic": a["role_oracle_generic"]["delta"], "Δ targeted": a["role_oracle_targeted"]["delta"]}
                  for w, a in ab.items()}).T
display(o.round(3))
t = pd.DataFrame({w: {k: v["auc"] for k, v in a["by_tier"].items()} for w, a in ab.items()}).T
t["pairs (agree / one human / tie-break)"] = " / ".join(str(ab["hand-set X2"]["by_tier"][k]["n_pairs"]) for k in ("humans_agree", "single_human", "llm_breaks_tie"))
t.round(3)

,role oracle Δ total,CI,Δ generic,Δ targeted
hand-set X2,-0.002476,"[-0.042, 0.031]",-0.011912,0.015923
logistic base (4),0.011725,"[-0.026, 0.042]",0.002442,0.029827


,humans_agree,llm_breaks_tie,single_human,pairs (agree / one human / tie-break)
hand-set X2,0.917,0.721,0.882,219 / 274 / 107
logistic base (4),0.912,0.739,0.884,219 / 274 / 107


The plan's rule for "worth its own plan" is a paired total ΔAUC of at least 0.02 with a CI above 0. The
role oracle does **not** reach it, so a perfect role flag is not shown to be where the headroom is.
The tier comparison is the largest gap in the table: the score does far worse on the pairs where the
two humans split and an LLM broke the tie. That is either a hard judgement or a doubtful label, and
numbers cannot separate the two.

### 2.4 Slices: where the score is weaker
Inversion rate over all (Fit, No Fit) pairs, sliced by an attribute of the **Fit-side** row.

In [8]:
rows = []
for name in ea["sets"][A2]["hand-set X2"]["slices"]:
    for s in (A2, D50S, D50H):
        for k, v in ea["sets"][s]["hand-set X2"]["slices"][name].items():
            rows.append({"slice": name, "value": k, "set": s, "pairs": v["n_pairs"], "inversion rate": v["inversion_rate"]})
sl = pd.DataFrame(rows).pivot_table(index=["slice", "value"], columns="set", values="inversion rate")[[A2, D50S, D50H]]
sl.round(3)

set                             A2 final (59 JDs)  D50 selection (70 JDs)  \
slice                  value                                                
CV length              long                 0.109                   0.096   
                       mid                  0.139                   0.129   
                       short                0.198                   0.199   
JD concepts <= 3       <= 3                 0.176                   0.084   
                       > 3                  0.136                   0.180   
any concept overlap    none                 0.220                   0.161   
                       overlap              0.098                   0.128   
experience years known known                0.170                   0.151   
                       unknown              0.139                   0.139   

set                             D50 held-out (30 JDs)  
slice                  value                           
CV length              long                     0.114  
                       mid                      0.195  
                       short                    0.149  
JD concepts <= 3       <= 3                     0.119  
                       > 3                      0.173  
any concept overlap    none                     0.178  
                       overlap                  0.135  
experience years known known                    0.104  
                       unknown                  0.172

Two patterns repeat across all three sets: pairs with **no concept overlap** are misordered more often,
and **long CVs** are misordered least. Whether the experience-years feature helps is not clear: pairs
with known years are not consistently better ordered. Other slices disagree between A2 and D50, so
they are not a basis for building anything yet.

## 3 · Why a person has to read some of these mistakes

The records above say *where* the score is wrong and *how often*. They cannot say *why*, for four
reasons.

1. **Attribution is arithmetic, not cause.** "TF-IDF pushed hardest" says a similarity term outweighed
   the others in that pair. It does not say whether the Fit CV was a real fit that the words missed
   (a transferable skill), a junior CV that fits on paper, or a label that is itself doubtful.
2. **Many mistakes sit where the role feature is silent.** On A2 that is most of them (section 2.1; on
   D50 it is about half). There the score has to separate two CVs that the role model treats alike, and
   that depends on seniority, core tools and transferable skills: things a reader recognises and a
   similarity score does not.
3. **The label tiers differ sharply** (section 2.3). Only a reader can tell whether a tie-break pair
   is a system error or a label that could have gone either way. If it is mostly the second, the
   ceiling for any score is lower than it looks and the right move is to stop optimising, not to
   build another feature.
4. **The LLM's note is not independent evidence.** The final set uses the same model's series as its
   secondary annotator (D43), so a note cannot confirm its own label, and keyword counts on the notes
   (notebook 20, check 3) are rough. A person's reason is the only independent reading we can get.

**What the review produces.** For each sampled mistake (up to 20 per culprit bucket, bucket hidden,
shuffled), the reviewer picks one reason from a list fixed before reading, with a short note:

| Tag | Reason | If it dominates, the next plan is about |
|---|---|---|
| A | role mismatch, but the Fit CV has transferable skills | cross-role matching / the role model |
| B | seniority or years are wrong | experience and seniority extraction |
| C | the No Fit CV matches on words but lacks the core requirement | core-requirement features (required vs preferred) |
| D | the label looks wrong | stop optimising; report the label ceiling |
| E | other | read the notes and re-plan |

These are error-reason tags, not relevance labels. They are stored in
`output/extraction/error-review-reasons.csv`, never enter `judgements.csv` or the final set, and hold
no document text (D57). The review informs what to build; it does not decide whether a lever clears
the 0.02 bar.